## Read all GLANCE data into dataframe

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import time

In [2]:
# read parquet file as pandas geodataframe. Downloaded from GLANCE website.
# https://source.coop/boston-university/bu-glance
# NB. Parquet file is 71 Mb
gdf = gpd.read_parquet('../Glance_data/bu_glance_training_dataV1.parquet')   # geodataframe
len(gdf)

1874995

In [3]:
gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [ ]:
# WGS84 (EPSG:4326) is a global CRS that uses latitude & longitude.
# Datum is WGS84, this is the ellipsoid. No projection - just lon and lat

In [5]:
len(gdf)

1874995

In [ ]:
# How many GLANCE samples per year
print('Samples per year')
sum=0
for i in range(1984,2023):
    y=np.sum((gdf['End_Year'] == i))
    print(i, '=', y)
    sum=sum+y
print('\ntotal samples =',sum)

**1.87 million global samples between 1984 and 2020**

In [13]:
gdf_no_change = gdf[(gdf['Change'] == False)]
len(gdf_no_change)

1486468

In [14]:
gdf = gdf_no_change.copy()
len(gdf)

1486468

In [15]:
gdf_same_year =gdf[(gdf['Start_Year'] == gdf['End_Year'])]
len(gdf_same_year)

601982

## Select samples by landcover Change and End_Year and then filter columns

In [ ]:
# landcover change == False and End_Year is > 2017
gdf_t = gdf[(gdf['Change'] == True) & (gdf['End_Year'] > 2017)]
gdf_t = gdf_t.filter(items=['Lat','Lon','Start_Year','End_Year','Continent_Code','Glance_Class_ID_level1','Glance_Class_ID_level2','geometry'])

In [ ]:
# landcover change == False and End_Year is > 2017
gdf = gdf[(gdf['Change'] == False) & (gdf['End_Year'] > 2017)]
gdf = gdf.filter(items=['Lat','Lon','Start_Year','End_Year','Continent_Code','Glance_Class_ID_level1','Glance_Class_ID_level2','geometry'])

In [ ]:
len(gdf), len(gdf_t), len(gdf)+len(gdf_t)

#### 1.27 million labelled points from 2017-2020 with no change in landcover  
#### 1.07 million labelled points from 2018-2020 with no change in landcover

## How many samples per Class and Year

In [ ]:
level1_class = ['Undefined(0)','Water(1)','Ice-Snow(2)','Developed(3)','Barren(4)','Trees(5)','Shrub(6)','Herbaceous(7)']
level2_class = ['Undefined(0)','Water(1)','Ice-Snow(2)','Developed(3)','Soil(4)','Rock(5)','Beach-Sand(6)', 'Deciduous(7)',
               'Evergreen(8)','Mixed(9)','Shrub(10)','Grassland(11)','Agriculture(12)','Moss(13)']    

In [ ]:
# How many samples per landcover class - level1
print('Samples Landcover Level1')
sum=0
for i in range(0,8):
    y=np.sum((gdf['Glance_Class_ID_level1'] == i))
    print(i, level1_class[i], '=', y)
    sum=sum+y
print('\ntotal samples =',sum)

In [ ]:
# How many samples per landcover class - level2
print('Samples Landcover Level2')
sum=0
for i in range(0,14):
    y=np.sum((gdf['Glance_Class_ID_level2'] == i))
    print(i, level2_class[i], '=', y)
    sum=sum+y
undefined = np.sum((gdf['Glance_Class_ID_level2'] == 0))
print('\ntotal samples =',sum)
print('total defined samples =',sum - undefined)

In [ ]:
# How many samples per year with no landcover change since 2017
print('Samples per year')
sum=0
for i in range(2017,2023):
    y=np.sum((gdf['End_Year'] == i))
    print(i, '=', y)
    sum=sum+y
print('\ntotal samples =',sum)

## Remove the 41 Ice-Snow samples

In [ ]:
# remove samples with Class_ID == 2
gdf = gdf[(gdf['Glance_Class_ID_level1'] != 2)]
len(gdf)

### Check Ice-Snow samples have been correctly removed

In [ ]:
# How many samples per landcover class - level1
print('Samples Landcover Level1')
sum=0
for i in range(0,8):
    y=np.sum((gdf['Glance_Class_ID_level1'] == i))
    print(i, level1_class[i], '=', y)
    sum=sum+y
print('\ntotal samples =',sum)

In [ ]:
# How many samples per landcover class - level2
print('Samples Landcover Level2')
sum=0
for i in range(0,14):
    y=np.sum((gdf['Glance_Class_ID_level2'] == i))
    print(i, level2_class[i], '=', y)
    sum=sum+y
print('\ntotal samples =',sum)

## Add column at zero location with sample number

In [ ]:
# insert column with row number
gdf.insert(loc=0, column='Sample', value=np.arange(len(gdf)))

## Re-number dataframe Level1 Classes

In [ ]:
level1_class = ['Undefined(0)','Water(1)','Ice-Snow(2)','Developed(3)','Barren(4)','Trees(5)','Shrub(6)','Herbaceous(7)']
level2_class = ['Undefined(0)','Water(1)','Ice-Snow(2)','Developed(3)','Soil(4)','Rock(5)','Beach-Sand(6)', 'Deciduous(7)',
               'Evergreen(8)','Mixed(9)','Shrub(10)','Grassland(11)','Agriculture(12)','Moss(13)']

level1_new = ['Water(0)','Developed(1)','Barren(2)','Trees(3)','Shrub(4)','Herbaceous(5)']  
level2_new = ['Water(0)','Developed(1)','Barren(2)','Trees(3)','Shrub(4)','Grassland(5)','Agriculture(6)','Undefined(7)']  

In [ ]:
# Glance Class ID Level 1
print('Class Level1 min and max')
print(np.min(gdf['Glance_Class_ID_level1']),np.max(gdf['Glance_Class_ID_level1']))

In [ ]:
# level1_class = ['Water(1)','Ice-Snow(2)','Developed(3)','Barren(4)','Trees(5)','Shrub(6)','Herbaceous(7)']

# level1_new = ['Water(0)','Developed(1)','Barren(2)','Trees(3)','Shrub(4)','Herbaceous(5)']

# initially there are no 0 or 2 classes. renumber from 1-7 to 0-5
# 0 -> 0
# 1 -> 0
# 2 -> 0
# 3 -> 1
# 4 -> 2
# 5 -> 3
# 6 -> 4
# 7 -> 5

new_list = [0,0,0,1,2,3,4,5]

gdf.iloc[:, 6] = gdf.iloc[:, 6].map(lambda x: new_list[x])

In [ ]:
# Glance Class ID Level 1
print('Class Level1 min and max')
print(np.min(gdf['Glance_Class_ID_level1']),np.max(gdf['Glance_Class_ID_level1']))

In [ ]:
# How many samples per landcover class - level1
print('Samples Landcover Level1')
sum=0
for i in range(0,6):
    y=np.sum((gdf['Glance_Class_ID_level1'] == i))
    print(i, level1_new[i], '=', y)
    sum=sum+y
print('\ntotal samples =',sum)

## Re-number dataframe Level2 Classes

In [ ]:
# Glance Class ID Level 2
print('Class Level2 min and max')
print(np.min(gdf['Glance_Class_ID_level2']),np.max(gdf['Glance_Class_ID_level2']))

In [ ]:
# level2_class = ['Undefined(0)','Water(1)','Ice-Snow(2)','Developed(3)','Soil(4)','Rock(5)','Beach-Sand(6)', 'Deciduous(7)',
# 'Evergreen(8)','Mixed(9)','Shrub(10)','Grassland(11)','Agriculture(12)','Moss(13)']

#level2_new = ['Water(0)','Developed(1)','Barren(2)', 'Trees(3)','Shrub(4)','Grassland(5)','Agriculture(6)','Undefined(7)']

# initially there are no 2 classes
# 0 -> 7 = undefined
# 1 -> 0
# 2 -> 0
# 3 -> 1
# 4 -> 2
# 5 -> 2
# 6 -> 2
# 7 -> 3
# 8 -> 3
# 9 -> 3
# 10 -> 4
# 11 -> 5
# 12 -> 6
# 13 -> 7

new_list = [7,0,0,1,2,2,2,3,3,3,4,5,6,7]

gdf.iloc[:, 7] = gdf.iloc[:, 7].map(lambda x: new_list[x])

In [ ]:
# How many samples per landcover class - level2
print('Samples Landcover Level2')
sum=0
for i in range(0,8):
    y=np.sum((gdf['Glance_Class_ID_level2'] == i))
    print(i, level2_new[i], '=', y)
    sum=sum+y
print('\ntotal samples =',sum)

In [ ]:
# Glance Class ID Level 2
print('Class Level2 min and max')
print(np.min(gdf['Glance_Class_ID_level2']),np.max(gdf['Glance_Class_ID_level2']))

## Add a Level3 column

In [ ]:
gdf_new = gdf.copy()
len(gdf_new)

In [ ]:
gdf_new.insert(loc=8, column='level3', value=99)

In [ ]:
gdf_new.head(2)

In [ ]:
# 1. Define the names and the condition value
col6_name = gdf_new.columns[6]
col7_name = gdf_new.columns[7]
col8_name = gdf_new.columns[8] 
x_value = 5                       # level-1 herbaceous


In [ ]:
col6_name, col7_name, col8_name, x_value

In [ ]:
# 2. Apply the conditional logic
gdf_new[col8_name] = np.where(
    gdf[col6_name] == x_value,  # Condition: Is Col 6 equal to x?
    gdf[col7_name],             # Value if TRUE (Column 3 == x): Use the value from Col 7
    gdf[col6_name]              # Value if FALSE (Column 3 != x): Use the value from Col 6
)

In [ ]:
gdf_new.head(2)

In [ ]:
# How many samples per landcover class - level3
print('Samples Landcover Level3')
sum=0
for i in range(0,8):
    y=np.sum((gdf_new['level3'] == i))
    print(i, level2_new[i], '=', y)
    sum=sum+y
print('\ntotal samples =',sum)

## Remove Undefined Samples (7)

In [ ]:
# remove samples with level3 == 7
gdf_new = gdf_new[(gdf_new['level3'] != 7)]
len(gdf_new)

In [ ]:
gdf_new.head()

## Save renumbered dataframe to parquet file

In [ ]:
# save dataframe to parquet file for rereading into Notebook
#gdf.to_parquet('../Glance_data/gdf_glance_all_2018_level3.parquet')
gdf_new.to_parquet('../Glance_data/gdf_glance_all_2018_level3.parquet')

In [ ]:
gdf_loaded = gpd.read_parquet('../Glance_data/gdf_glance_all_2018_level3.parquet')

In [ ]:
len(gdf_loaded)

In [ ]:
gdf_loaded.head(2)

In [ ]:
print('Samples per Continent')
sum=0
for i in range(0,7):
    y=np.sum((gdf_loaded['Continent_Code'] == i))
    print(i, '=', y)
    sum=sum+y
print('\ntotal samples =',sum)